# Day 1: LLM Foundations + LangChain Core

Goal: understand tokens/temperature/roles, LangChain's Runnable/LCEL model,
and structured output via Pydantic — the building blocks for the
zero-hallucination RAG pipeline.

In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()
print("Loaded environment variables from .env file")

Loaded environment variables from .env file


In [5]:
# This pattern — LLM output validated against a schema — is the backbone
# of the citation-backed generation step we'll build on Day 4.
class DocumentSummary(BaseModel):
    title: str = Field(description="A short title for the summary")
    key_points: list[str] = Field(description="3 bullet point key takeaways")
    confidence: str = Field(description="high, medium, or low")

DocumentSummary.model_json_schema()  # sanity check: view the schema LangChain will enforce

{'properties': {'title': {'description': 'A short title for the summary',
   'title': 'Title',
   'type': 'string'},
  'key_points': {'description': '3 bullet point key takeaways',
   'items': {'type': 'string'},
   'title': 'Key Points',
   'type': 'array'},
  'confidence': {'description': 'high, medium, or low',
   'title': 'Confidence',
   'type': 'string'}},
 'required': ['title', 'key_points', 'confidence'],
 'title': 'DocumentSummary',
 'type': 'object'}

In [6]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# with_structured_output wraps the model so its response is parsed
# and validated into our Pydantic class, not returned as raw text
structured_model = model.with_structured_output(DocumentSummary)

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarize text concisely and objectively."),
    ("user", "{text}")
])

In [8]:
# The "|" is LCEL: it composes two Runnables (prompt, structured_model)
# into a single Runnable pipeline.
chain = prompt | structured_model

type(chain)  # confirm it's a RunnableSequence

langchain_core.runnables.base.RunnableSequence

In [9]:
sample_text = """
LangChain is a framework for building applications powered by large language
models. It provides abstractions for prompts, chat models, and output parsers,
and lets you compose them into pipelines using LCEL. LangGraph extends this
with state, branching, and cycles for agentic workflows.
"""

result = chain.invoke({"text": sample_text})

print(result)
print(type(result))

title='Overview of LangChain and LangGraph' key_points=['LangChain is a framework for creating applications using large language models.', 'It offers abstractions for prompts, chat models, and output parsers, allowing for pipeline composition.', 'LangGraph enhances LangChain with features like state management, branching, and cycles for more complex workflows.'] confidence='high'
<class '__main__.DocumentSummary'>


In [10]:
# Because this is a validated Pydantic object (not a raw string),
# you can access fields directly — no regex, no manual parsing.
print("Title:", result.title)
print("Key points:")
for point in result.key_points:
    print(" -", point)
print("Confidence:", result.confidence)

Title: Overview of LangChain and LangGraph
Key points:
 - LangChain is a framework for creating applications using large language models.
 - It offers abstractions for prompts, chat models, and output parsers, allowing for pipeline composition.
 - LangGraph enhances LangChain with features like state management, branching, and cycles for more complex workflows.
Confidence: high


## Provider-swap demo

The point of the `Runnable` abstraction: swapping the underlying model
provider should require zero changes to `prompt` or the chain composition.
This cell won't run without real Azure credentials — that's fine, we're
just confirming the shape of the swap.

In [11]:
# from langchain_openai import AzureChatOpenAI

# # Not run for real yet — provider is still TBD (per learning-plan.md).
# # This just demonstrates: same prompt, same chain shape, different model.
# azure_model = AzureChatOpenAI(
#     azure_deployment="your-deployment-name",
#     api_version="2024-10-21",
#     temperature=0,
# )
# azure_structured = azure_model.with_structured_output(DocumentSummary)

# azure_chain = prompt | azure_structured
# # azure_chain.invoke({"text": sample_text})  # uncomment once Azure creds exist

# print("azure_chain built — prompt object is unchanged:", prompt is prompt)

## Takeaway

_Write 2-3 sentences in your own words: what does `Runnable`/LCEL actually
buy you, and why does `with_structured_output` matter for a
zero-hallucination system?_